# Setup & Initialization

In [ ]:
import random
import pyspark
from delta.tables import DeltaTable, IdentityGenerator
from pyspark.sql import SparkSession, functions as F
import ConnectionConfig as cc
from importlib import reload
from pyspark.sql.types import LongType, StringType, TimestampType, IntegerType, BooleanType, BinaryType

reload(cc)
from pyspark.sql.functions import *

print(cc.__file__)

cc.setupEnvironment()
cc.listEnvironment()

In [ ]:
spark = cc.startLocalCluster("dimTreasureInit")
spark.getActiveSession()

## Reading the source table

In [ ]:
cc.set_connectionProfile("catchem")

print(cc.create_jdbc())

In [ ]:
# Load DataFrames
df_operational_stage = cc.read_postgres_without_partition(spark, "treasure_stages")

print('size: ', df_operational_stage.count())

In [ ]:
df_aggregate_stage = (
    df_operational_stage
    .groupBy("treasure_id")
    .agg(F.count("stages_id").alias("count"))
    .orderBy(F.col("count").desc())
)

df_aggregate_stage.show(10,truncate=False)

## Creating DataFrame Of Treasure Dimension

In [ ]:
max_stage_count = (
    df_aggregate_stage
    .agg(F.max("count").alias("max_count"))
    .first()["max_count"]
)

max_stage_count


In [ ]:
df_difficulty = spark.range(0, 5).withColumnRenamed("id", "difficulty")

df_terrain = spark.range(0, 5).withColumnRenamed("id", "terrain")

df_stage_count = spark.range(1, max_stage_count + 1).withColumnRenamed("id", "stage_count")

# Cartesian Product to get all combinations of difficulty, terrain, and stage_count
df_treasure_combined = (
    df_difficulty.crossJoin(df_terrain)
    .crossJoin(df_stage_count)
)

df_treasure_combined = df_treasure_combined.orderBy("difficulty", "terrain", "stage_count")

# Cast your columns to IntegerType to match the table definition
df_treasure_final = df_treasure_combined.select(
    F.col("difficulty").cast(IntegerType()),
    F.col("terrain").cast(IntegerType()),
    F.col("stage_count").cast(IntegerType())
)

df_treasure_final.show(truncate=False)

## Deleting the dimension table if it already exists

In [ ]:
cc.delete_delta_table(spark, "dim_treasure")

## Creating the dimension table

In [ ]:
#Initialize the table with the identity column

DeltaTable.create(spark) \
    .tableName("dim_treasure") \
    .addColumn("treasure_sk", LongType(), nullable=False, generatedAlwaysAs=IdentityGenerator(0, 1)) \
    .addColumn("difficulty", IntegerType(), nullable=False) \
    .addColumn("terrain", IntegerType(), nullable=False) \
    .addColumn("stage_count", IntegerType(), nullable=False) \
    .property("delta.feature.identityColumns", "supported") \
    .execute()

## Populating the dimension table


In [ ]:
df_treasure_final.write.format("delta").mode("append").saveAsTable("dim_treasure")

In [ ]:
# Delete parquet file if it already exists
cc.delete_parquet_table("./parquet/dim_treasure_parquet")

In [ ]:
# Save as Parquet format

df = spark.read.format("delta").load("spark-warehouse/dim_treasure")

df.coalesce(1).write.format("parquet").save("./parquet/dim_treasure_parquet")

## Checking the results

In [ ]:
spark.read.format("delta").load("spark-warehouse/dim_treasure").show(10, truncate=False)

In [ ]:
spark.read.format("delta").load("spark-warehouse/dim_treasure") \
    .orderBy(F.col("stage_count").desc()) \
    .show(truncate=False)

In [ ]:
spark.stop()

In [ ]:
%reset